#### Name: Nguyen Thi Thu Trang
#### Class: DSEB65A
#### Student ID: 11230595
#### Homework 5

## **Theoretical Exercise: Processing Workflow and Data Leakage**

Assume you have a dataset containing both missing values and outliers. Someone proposes the following processing workflow:

- **Step 1:** Remove outliers from the entire dataset.  
- **Step 2:** Use SimpleImputer to fill in missing values on the entire dataset.
- **Step 3:** Split the cleaned dataset into a training set and a testing set.
- **Step 4:** Train a model on the training set.


Answer the following questions:

1. **Error Analysis:** Where did the process go wrong, and why does it cause data leakage?  
The process goes wrong when people try to prepare data first, then split later. The right process is always follows the golden rule: split first, prepare later because when information from the test set is accidentally "leaked" into the model training process, so model uses information during training that wouldn't be available at the time of prediction, it occurs data leakage.

2. **Consequences:** How will data leakage affect the model's evaluation metrics?  
Leakage causes a predictive model to look accurate until deployed in its use case; then, it will yield inaccurate results, leading to poor decision-making and false insights.

3. **The Correct Workflow:** Outline the correct sequence of steps to avoid data leakage.  
The correct steps are:
    1. Split: Split raw data into train and test sets.
    2. Fit: Learn the transform parameters ONLY on the TRAIN set.
    3. Transform: Apply the transform to both the train and test sets.
    4. Train the model.

4. **Role of the Pipeline:** How does a Scikit-learn Pipeline help solve this problem?  
The pipeline helps to ensure the order of the opearions and prevent some manual mistakes. For example, the preprocessing will always be applied in the training data set when use the Scikit-learn Pipeline, so it helps prevent the data leakage.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import io

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

## **Practical Exercise 1: Oil Spill Dataset - Comparing the Effectiveness of a Comprehensive Cleaning Workflow**

### 1. Preparation

#### 1.1 Load the data

In [3]:
url_lab1 = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/oil-spill.csv'
df_lab1 = pd.read_csv(url_lab1, header=None)
df_lab1.head()

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,1,2558,1506.09,456.63,90,6395000.0,40.88,7.89,29780.0,0.19,...,2850.00,1000.00,763.16,135.46,3.73,0,33243.19,65.74,7.95,1
1,2,22325,79.11,841.03,180,55812500.0,51.11,1.21,61900.0,0.02,...,5750.00,11500.00,9593.48,1648.80,0.60,0,51572.04,65.73,6.26,0
2,3,115,1449.85,608.43,88,287500.0,40.42,7.34,3340.0,0.18,...,1400.00,250.00,150.00,45.13,9.33,1,31692.84,65.81,7.84,1
3,4,1201,1562.53,295.65,66,3002500.0,42.40,7.97,18030.0,0.19,...,6041.52,761.58,453.21,144.97,13.33,1,37696.21,65.67,8.07,1
4,5,312,950.27,440.86,37,780000.0,41.43,7.03,3350.0,0.17,...,1320.04,710.63,512.54,109.16,2.58,0,29038.17,65.66,7.35,0


#### 1.2 Remove duplicates

In [4]:
# Check for duplicate rows
print(f"\nNumber of duplicate rows: {df_lab1.duplicated().sum()}")

# Remove duplicate rows
df1_no_dup = df_lab1.drop_duplicates()
print("\nData after removing duplicate rows:")
print(df1_no_dup.head())


Number of duplicate rows: 0

Data after removing duplicate rows:
   0      1        2       3    4           5      6     7        8     9   \
0   1   2558  1506.09  456.63   90   6395000.0  40.88  7.89  29780.0  0.19   
1   2  22325    79.11  841.03  180  55812500.0  51.11  1.21  61900.0  0.02   
2   3    115  1449.85  608.43   88    287500.0  40.42  7.34   3340.0  0.18   
3   4   1201  1562.53  295.65   66   3002500.0  42.40  7.97  18030.0  0.19   
4   5    312   950.27  440.86   37    780000.0  41.43  7.03   3350.0  0.17   

   ...       40        41       42       43     44  45        46     47    48  \
0  ...  2850.00   1000.00   763.16   135.46   3.73   0  33243.19  65.74  7.95   
1  ...  5750.00  11500.00  9593.48  1648.80   0.60   0  51572.04  65.73  6.26   
2  ...  1400.00    250.00   150.00    45.13   9.33   1  31692.84  65.81  7.84   
3  ...  6041.52    761.58   453.21   144.97  13.33   1  37696.21  65.67  8.07   
4  ...  1320.04    710.63   512.54   109.16   2.58   0  2903

### 2. Compare two approaches

#### Approach A: Train Pipeline 1 (Baseline) on the original training set.

In [5]:
# Split into training and testing sets
X = df1_no_dup.iloc[:, :-1]
y = df1_no_dup.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape, " Test set:", X_test.shape)

# Pipeline 1
pipe1 = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", LogisticRegression(max_iter=1000))
])

pipe1.fit(X_train, y_train)
y_pred_a = pipe1.predict(X_test)
acc_a = accuracy_score(y_test, y_pred_a)

print("\nApproach A (Baseline) Accuracy:", acc_a)

Training set: (749, 49)  Test set: (188, 49)

Approach A (Baseline) Accuracy: 0.9627659574468085


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### Approach B: Train Pipeline 2 (Basic Cleaning) on the outlier-cleaned training set.

In [6]:
# Apply the Outlier Handling Workflow (IQR) on the training set
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1

lower_iqr, upper_iqr = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
mask = ~((X_train < lower_iqr) | (X_train > upper_iqr)).any(axis=1)

X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

print("Training set size before outlier removal:", len(X_train))
print("Training set size after outlier removal:", len(X_train_clean))

# Pipeline 2
pipe2 = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("var_thresh", VarianceThreshold(threshold=0.05)),
    ("model", LogisticRegression(max_iter=1000))
])

pipe2.fit(X_train_clean, y_train_clean)
y_pred_b = pipe2.predict(X_test)
acc_b = accuracy_score(y_test, y_pred_b)

print("Approach B (Comprehensive Cleaning) Accuracy:", acc_b)

Training set size before outlier removal: 749
Training set size after outlier removal: 332
Approach B (Comprehensive Cleaning) Accuracy: 0.8936170212765957


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 3. Evaluation

In [7]:
print("Approach A (Baseline) Accuracy:", acc_a)
print("Approach B (Basic Cleaning) Accuracy:", acc_b)

Approach A (Baseline) Accuracy: 0.9627659574468085
Approach B (Basic Cleaning) Accuracy: 0.8936170212765957


In [8]:
results = pd.DataFrame({
    "Approach": ["A. Baseline", "B. Comprehensive Cleaning"],
    "Accuracy": [acc_a, acc_b]
})
print("\nComparison of Approaches:")
print(results)


Comparison of Approaches:
                    Approach  Accuracy
0                A. Baseline  0.962766
1  B. Comprehensive Cleaning  0.893617


## **Practical Exercise 2: Housing Dataset - Comparing Outlier Handling Methods**

### 1. Preparation

#### 1.1 Load the data

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [10]:
column_names = [
    "CRIM",    # Tội phạm bình quân đầu người theo thị trấn
    "ZN",      # Tỷ lệ đất ở > 25,000 sq.ft
    "INDUS",   # Tỷ lệ diện tích cho doanh nghiệp phi bán lẻ
    "CHAS",    # Biến giả sông Charles (=1 nếu gần sông, 0 nếu không)
    "NOX",     # Nồng độ oxit nitơ (phần triệu)
    "RM",      # Số phòng trung bình mỗi căn hộ
    "AGE",     # % căn hộ xây dựng trước 1940
    "DIS",     # Khoảng cách bình quân đến 5 trung tâm việc làm
    "RAD",     # Chỉ số khả năng tiếp cận đường cao tốc
    "TAX",     # Thuế bất động sản
    "PTRATIO", # Tỷ lệ học sinh/giáo viên
    "B",       # 1000(Bk - 0.63)^2, với Bk % dân da đen
    "LSTAT",   # % dân có địa vị kinh tế xã hội thấp
    "MEDV"     # Giá trị trung vị của nhà (ngàn USD)
]
url_lab2 = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/housing.csv"
df_lab2 = pd.read_csv(url_lab2, header=None, names=column_names)
df_lab2
df_lab2.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


### 2. Compare four approaches

#### Approach A (Baseline): Train Pipeline 2 (Basic Cleaning) with a RandomForestRegressor model on the original training set.

In [11]:
# Split training and testing set
X = df_lab2.drop("MEDV", axis=1)
y = df_lab2["MEDV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create pipeline A
pipeA = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])

pipeA.fit(X_train, y_train)
y_predA = pipeA.predict(X_test)
mseA = mean_squared_error(y_test, y_predA)
print(f"MSE of Approach A (Baseline) is {mseA}")

MSE of Approach A (Baseline) is 7.912745333333333


#### Approach B (IQR): Apply the Outlier Handling Workflow (IQR), then train Pipeline 2 on the cleaned training set

In [12]:
# Apply the Outlier Handling Workflow (IQR) on the training set
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1

lower_iqr, upper_iqr = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
mask = ~((X_train < lower_iqr) | (X_train > upper_iqr)).any(axis=1)

X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

# Pipeline B
pipeB = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])

pipeB.fit(X_train_clean, y_train_clean)
y_predB = pipeB.predict(X_test)
mseB = mean_squared_error(y_test, y_predB)

print(f"MSE of Approach B (IQR) is {mseB}")

MSE of Approach B (IQR) is 17.252275970588265


#### Approach C (Standard Deviation - Std): Apply the Outlier Handling Workflow (Std) with a threshold of 3 standard deviations, then train Pipeline 2 on the cleaned training set.

In [13]:
# Apply the Outlier Handling Workflow (Std) with a threshold of 3 standard deviations
def remove_outliers_std(X, y, threshold=3):
    df = X.copy()
    df["MEDV"] = y
    z_scores = ((df - df.mean()) / df.std()).abs()
    mask = (z_scores < threshold).all(axis=1)
    return df.loc[mask].drop("MEDV", axis=1), df.loc[mask]["MEDV"]

X_train_std, y_train_std = remove_outliers_std(X_train, y_train)

# Pipeline C
pipeC = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])

pipeC.fit(X_train_std, y_train_std)
y_predC = pipeC.predict(X_test)
mseC = mean_squared_error(y_test, y_predC)

print(f"MSE of Approach C (Standard Deviation - Std) is {mseC}")

MSE of Approach C (Standard Deviation - Std) is 12.881737754901957


#### Approach D (LOF): Apply the Outlier Handling Workflow (LOF), then train Pipeline 2 on the cleaned training set.

In [14]:
from sklearn.neighbors import LocalOutlierFactor

def remove_outliers_lof(X, y, contamination=0.05):
    lof = LocalOutlierFactor(contamination=contamination)
    yhat = lof.fit_predict(X)
    mask = yhat != -1
    return X[mask], y[mask]

X_train_lof, y_train_lof = remove_outliers_lof(X_train, y_train)

# Pipeline C
pipeD = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor(random_state=42))
    ])

pipeD.fit(X_train_lof, y_train_lof)
y_predD = pipeD.predict(X_test)
mseD = mean_squared_error(y_test, y_predD)

print(f"MSE of Approach D (LOF) is {mseD}")

MSE of Approach D (LOF) is 9.002561686274504


### 3. Evaluation

In [15]:
print(f"MSE of Approach A (Baseline) is {mseA}")
print(f"MSE of Approach B (IQR) is {mseB}")
print(f"MSE of Approach C (Standard Deviation - Std) is {mseC}")
print(f"MSE of Approach D (LOF) is {mseD}")

MSE of Approach A (Baseline) is 7.912745333333333
MSE of Approach B (IQR) is 17.252275970588265
MSE of Approach C (Standard Deviation - Std) is 12.881737754901957
MSE of Approach D (LOF) is 9.002561686274504


### 4. Conclusion

In [16]:
results = pd.DataFrame({
    "Approach": ["Baseline", "IQR", "Std (3σ)", "LOF"],
    "MSE": [mseA, mseB, mseC, mseD]
})
print(results)

   Approach        MSE
0  Baseline   7.912745
1       IQR  17.252276
2  Std (3σ)  12.881738
3       LOF   9.002562


### **Practical Exercise 3: Horse Colic Dataset - Comparing Imputation Methods**

In [17]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

### 1. Preparation

#### 1.1 Load the data

In [18]:
url_lab3 = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/horse-colic.csv"

col_names = [
    "surgery", "age", "hospital_number", "rectal_temp", "pulse",
    "respiratory_rate", "temp_extremities", "peripheral_pulse",
    "mucous_membrane", "capillary_refill", "pain", "peristalsis",
    "abdominal_distension", "nasogastric_tube", "nasogastric_reflux",
    "nasogastric_reflux_ph", "rectal_exam_feces", "abdomen",
    "packed_cell_volume", "total_protein", "abdomocentesis_appearance",
    "abdomocentesis_total_protein", "outcome", "surgical_lesion",
    "lesion_1", "lesion_2", "lesion_3", "lesion_4"
]

df_lab3 = pd.read_csv(url_lab3, header=None, names=col_names, na_values="?")

df_lab3.head()

,surgery,age,hospital_number,rectal_temp,pulse,respiratory_rate,temp_extremities,peripheral_pulse,mucous_membrane,capillary_refill,...,packed_cell_volume,total_protein,abdomocentesis_appearance,abdomocentesis_total_protein,outcome,surgical_lesion,lesion_1,lesion_2,lesion_3,lesion_4
0,2.0,1,530101,38.5,66.0,28.0,3.0,3.0,NaN,2.0,...,45.0,8.4,NaN,NaN,2.0,2,11300,0,0,2
1,1.0,1,534817,39.2,88.0,20.0,NaN,NaN,4.0,1.0,...,50.0,85.0,2.0,2.0,3.0,2,2208,0,0,2
2,2.0,1,530334,38.3,40.0,24.0,1.0,1.0,3.0,1.0,...,33.0,6.7,NaN,NaN,1.0,2,0,0,0,1
3,1.0,9,5290409,39.1,164.0,84.0,4.0,1.0,6.0,2.0,...,48.0,7.2,3.0,5.3,2.0,1,2208,0,0,1
4,2.0,1,530255,37.3,104.0,35.0,NaN,NaN,6.0,2.0,...,74.0,7.4,NaN,NaN,2.0,2,4300,0,0,2


#### 1.2 Split into training and testing sets

In [19]:
df_lab3 = df_lab3.dropna(subset=["outcome"])

X = df_lab3.drop("outcome", axis=1)
y = df_lab3["outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 2. Basic Cleaning

In [20]:
# Apply the Outlier Handling Workflow (IQR)
Q1 = X_train.quantile(0.25)
Q3 = X_train.quantile(0.75)
IQR = Q3 - Q1

lower_iqr, upper_iqr = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
mask = ~((X_train < lower_iqr) | (X_train > upper_iqr)).any(axis=1)

X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

### 3. Compare three Pipelines

#### Approach A: Train Pipeline 2 (Basic Cleaning) with SimpleImputer(strategy='median').

In [21]:
# Pipeline A
pipeA = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

#### Approach B: Train Pipeline 3a (KNN Imputation).

In [22]:
pipeB = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

#### Approach C: Train Pipeline 3b (Iterative Imputation).

In [23]:
pipeC = Pipeline([
    ("imputer", IterativeImputer(random_state=42)),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])

### 4. Evaluation and Conclusion

In [24]:
pipelines = {"Median": pipeA, "KNN": pipeB, "Iterative": pipeC}
results = []

for name, pipe in pipelines.items():
    pipe.fit(X_train_clean, y_train_clean)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append([name, acc])
    print(f"\n{name} Imputation:")
    print(classification_report(y_test, y_pred))


Median Imputation:
              precision    recall  f1-score   support

         1.0       0.77      0.83      0.80        36
         2.0       0.73      0.73      0.73        15
         3.0       0.17      0.11      0.13         9

    accuracy                           0.70        60
   macro avg       0.56      0.56      0.56        60
weighted avg       0.67      0.70      0.68        60


KNN Imputation:
              precision    recall  f1-score   support

         1.0       0.76      0.78      0.77        36
         2.0       0.76      0.87      0.81        15
         3.0       0.17      0.11      0.13         9

    accuracy                           0.70        60
   macro avg       0.56      0.59      0.57        60
weighted avg       0.67      0.70      0.68        60


Iterative Imputation:
              precision    recall  f1-score   support

         1.0       0.74      0.78      0.76        36
         2.0       0.79      0.73      0.76        15
         3.0   

In [25]:
results_df = pd.DataFrame(results, columns=["Approach", "Accuracy"])
print(results_df)

    Approach  Accuracy
0     Median  0.700000
1        KNN  0.700000
2  Iterative  0.666667
